# Shardy：先传播分片，再生成局部计算

本 Notebook 复查源码 wheel 003 上保存的双 CPU 结果。当前执行是 `REPLAY-OFFLINE`；native MLIR/HLO 解析已由固定镜像中的 reader 完成，这里核对其归档与原始文件指纹，不用宿主旧 wheel 重新解释 IR。

见 [说明与复跑命令](shardy-round-trip.md)。原始大文件位于仓库忽略目录。

In [1]:
from pathlib import Path
import json,sys
import numpy as np
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"upstream-sources.lock").exists())
sys.path.insert(0,str(ROOT/"research/software-stack"))
from verify_research import read_json,local_path,sha256
from cpu_executable_parser import inventory
from verify_shardy import check_propagation,check_partition,selftest
result=read_json(ROOT/"research/software-stack/shardy-results.json")
source=ROOT/"artifacts/jax-stack/shardy-runtime-001"
assert result["evidence_level"]=="REPLAY-OFFLINE"
for case in result["cases"]:
    assert inventory(source/case["mode"],"RUN-CPU")==case["runtime"]
    for artifact in case["stage_artifacts"].values():
        assert sha256(local_path(artifact["path"]))==artifact["sha256"]
print({"modes":[c["mode"] for c in result["cases"]],"artifacts":sum(c["runtime"]["artifact_count"] for c in result["cases"])})

{'modes': ['shardy', 'gspmd'], 'artifacts': 65}


## 传播改变属性，结果仍是全局形状

查看 dot/add/maximum。这里展示 native reader 保存的 operation/attribute 视图；`[8,12]` 仍是全局结果。

In [2]:
sdy=result["cases"][0]
check_propagation(sdy["mlir"])
from verify_shardy import named
for name in ["stablehlo.dot","stablehlo.add","stablehlo.maximum"]:
    before=named(sdy["mlir"]["before"],name)
    after=named(sdy["mlir"]["after"],name)
    print(name,before["attributes"].get("sdy.sharding"),"→",after["attributes"].get("sdy.sharding"),after["results"])
print(named(sdy["mlir"]["after"],"stablehlo.dot")["attributes"]["sdy.sharding_rule"])

stablehlo.dot None → #sdy.sharding_per_value<[<@mesh, [{"rows"}, {}]>]> ['tensor<8x12xf32>']
stablehlo.add None → #sdy.sharding_per_value<[<@mesh, [{"rows"}, {}]>]> ['tensor<8x12xf32>']
stablehlo.maximum None → #sdy.sharding_per_value<[<@mesh, [{"rows"}, {}]>]> ['tensor<8x12xf32>']
#sdy.op_sharding_rule<([i, k], [k, j])->([i, j]) {i=8, j=12, k=16} reduction={k}>


## 后续 SPMD 分区才形成局部 dot

两个模式都将输出从 `[8,12]` 分为每 CPU `[4,12]`。此处没有沿 k 分片，也未测量通信或性能。

In [3]:
for case in result["cases"]:
    check_partition(case["hlo"])
    for stage in ["propagated","partitioned"]:
        dot=next(n for n in case["hlo"][stage]["nodes"].values() if n["opcode"]=="kDot")
        print(case["mode"],stage,dot["text"].split(", metadata=")[0])
    print(case["output_shards"])
for pair in result["single_partition_reference"]["pairs"]:
    assert local_path(pair["before"]["path"]).read_bytes()==local_path(pair["after"]["path"]).read_bytes()
print("旧单 CPU 的四对 shardy-xla 边界相同，不计为分片传播证据。")

shardy propagated %dot_general.0 = f32[8,12]{1,0} dot(%a.0, %w.0), lhs_contracting_dims={1}, rhs_contracting_dims={0}, operand_precision={highest,highest}, sharding={devices=[2,1]<=[2]}
shardy partitioned %dot = f32[4,12]{1,0} dot(%param, %param.1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, operand_precision={highest,highest}
[{'device_id': 0, 'index': [[0, 4, None], [None, None, None]], 'shape': [4, 12]}, {'device_id': 1, 'index': [[4, 8, None], [None, None, None]], 'shape': [4, 12]}]
gspmd propagated %dot_general.1 = f32[8,12]{1,0} dot(%a.1, %w.1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, operand_precision={highest,highest}, sharding={devices=[2,1]<=[2]}
gspmd partitioned %dot = f32[4,12]{1,0} dot(%param, %param.1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, operand_precision={highest,highest}
[{'device_id': 0, 'index': [[0, 4, None], [None, None, None]], 'shape': [4, 12]}, {'device_id': 1, 'index': [[4, 8, None], [None, None, None]], 'shape': [4, 12]}]


## 六次执行与 NumPy 独立参考

分别检查全局输出及实际局部分片。两种算法相同输出不代表任意模型具有相同策略或性能。

In [4]:
from verify_shardy import check_shards
for case in result["cases"]:
    capture=source/case["mode"]
    with np.load(capture/"inputs.npz",allow_pickle=False) as x,np.load(capture/"outputs.npz",allow_pickle=False) as y:
        reference=np.maximum(x["a"].astype(np.float64)@x["w"].astype(np.float64)+x["bias"].astype(np.float64),0)
        errors=[]
        for i in range(3):
            np.testing.assert_allclose(y[f"result_{i}"],reference,rtol=2e-5,atol=2e-5)
            errors.append(float(np.max(np.abs(y[f"result_{i}"]-reference))))
        check_shards(read_json(capture/"summary.json"),y,reference)
        assert errors==case["max_absolute_errors"]
        print(case["mode"],errors)

shardy [3.483544636084801e-08, 3.483544636084801e-08, 3.483544636084801e-08]
gspmd [3.483544636084801e-08, 3.483544636084801e-08, 3.483544636084801e-08]


## 错误证据拒绝与边界

反例是归档检查，不是额外设备实验。混合 IR 自动 fallback、V3、tuple/alias 恢复和真实 TPU 未完成运行验收。

In [5]:
checks=selftest(result,source)
assert checks==result["negative_tests"]
print({"rejected_invalid_cases":len(checks),"runtime_scope":"two CPU devices, one process","notebook_scope":"archive revalidation"})

{'rejected_invalid_cases': 9, 'runtime_scope': 'two CPU devices, one process', 'notebook_scope': 'archive revalidation'}
